In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import plotly.express as px
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
import time
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTENC
import shap

# Преобработка

In [ ]:
df_lep1 = pd.read_excel('Протяженность, напряжение.xlsx', sheet_name='кабельные лэп')
df_lep1.head()

In [ ]:
df_lep1.info()

In [ ]:
for col in df_lep1.columns:
    try:
        uniques = df_lep1[col].unique()
    except TypeError:
        uniques = df_lep1[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_lep1.isna().sum()

In [ ]:
df_lep1["Вид деятельности"] = (
    df_lep1["Вид деятельности"].replace(0, pd.NA)
)

df_lep1["Вид деятельности"] = (
    df_lep1.groupby("Субъект РФ")["Вид деятельности"]
    .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else pd.NA))
)

In [ ]:
df_lep1.isna().sum()

In [ ]:
df_lep1[df_lep1['Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС'] == 0]

In [ ]:
df_lep1["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"] = df_lep1.groupby(["Эксплуатирующая организация", "Вид деятельности", "Субъект РФ"])["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"].transform(
    lambda x: x.replace(0, pd.NA)  
).transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df_lep1[df_lep1['Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС'] == 0]

In [ ]:
df_lep1.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_lep1["Лет в эксплуатации"] = (
    today.year - df_lep1["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_lep1["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_lep1["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_lep1["Дата ввода в экспл.в БУ"].dt.day)))
)

df_lep1[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_lep1 = df_lep1.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_lep1.columns = ['класс', 'эксплуатирующая организация', 'вид деятельности',
       'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб',
       'протяженность, м', 'напряжение, кв', 'лет в эксплуатации']

In [ ]:
df_lep1 = df_lep1.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

In [ ]:
df_lep2 = pd.read_excel('Протяженность, напряжение.xlsx', sheet_name='лэп')
df_lep2.head()

In [ ]:
df_lep2.info()

In [ ]:
for col in df_lep2.columns:
    try:
        uniques = df_lep2[col].unique()
    except TypeError:
        uniques = df_lep2[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_lep2.isna().sum()

In [ ]:
df_lep2["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"].describe()

In [ ]:
df_lep2["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"] = df_lep2.groupby(["Эксплуатирующая организация", "Вид деятельности", "Субъект федерации"])["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"].transform(
    lambda x: x.replace(0, pd.NA)  
).transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df_lep2["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"].describe()

In [ ]:
df_lep2["Напряжение, кВ"].describe()    

In [ ]:
df_lep2["Напряжение, кВ"] = df_lep2.groupby(["Эксплуатирующая организация", "Вид деятельности", "Субъект федерации"])["Напряжение, кВ"].transform(
    lambda x: x.replace(0, pd.NA)  
).transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df_lep2["Напряжение, кВ"].describe()    

In [ ]:
df_lep2.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_lep2["Лет в эксплуатации"] = (
    today.year - df_lep2["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_lep2["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_lep2["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_lep2["Дата ввода в экспл.в БУ"].dt.day)))
)

df_lep2[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_lep2 = df_lep2.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_lep2.columns = ['класс', 'эксплуатирующая организация', 'вид деятельности',
       'субъект РФ', 'стоимость в ценах 2023 года без НДС, руб',
       'протяженность, м', 'напряжение, кв', 'лет в эксплуатации']

In [ ]:
df_lep2 = df_lep2.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

# Объединение данных

In [ ]:
df_lep = pd.concat([df_lep1, df_lep2], ignore_index=True)

In [ ]:
df_lep.info()

In [ ]:
col = 'напряжение, кв'

df_lep[col] = (
    df_lep[col]
    .astype(str)
    .str.replace(r'[^0-9.,]', '', regex=True)  
    .str.replace(',', '.', regex=False)
    .astype(float)       
)

print(df_lep[col].head())
print(df_lep[col].dtype)

In [ ]:
col = 'стоимость в ценах 2023 года без НДС, руб'

df_lep[col] = (
    df_lep[col]
    .astype(str)
    .str.replace(r'[^0-9.,]', '', regex=True)  
    .str.replace(',', '.', regex=False)
    .astype(float)       
)

print(df_lep[col].head())
print(df_lep[col].dtype)

In [ ]:
df_lep.info()

In [ ]:
# Справочник «субъект РФ → код» хранится отдельно и не публикуется, чтобы коды на графиках нельзя было расшифровать.
import json
region_mapping = json.load(open("data/region_mapping.json", encoding="utf-8"))

In [ ]:
df_lep['субъект РФ'] = df_lep['субъект РФ'].map(region_mapping)

In [ ]:
df_lep['субъект РФ'].unique()

In [ ]:
activity_mapping = {
    "Добыча газа и газового конденсата": "Деятельность_01",
    "Транспортировка газа": "Деятельность_02",
    "Прочие виды деятельности": "Деятельность_03",
    "Подземное хранение газа": "Деятельность_04",
    "Переработка углеводородов": "Деятельность_05",
    "Производство СПГ": "Деятельность_06",
    "Газораспределение": "Деятельность_07"
}

In [ ]:
df_lep["вид деятельности"] = df_lep["вид деятельности"].map(activity_mapping)

In [ ]:
df_lep["вид деятельности"].unique()

In [ ]:
# Справочник «организация → код» не публикуется: названия эксплуатирующих организаций — внутренние данные.
import json
org_mapping = json.load(open("data/org_mapping.json", encoding="utf-8"))

In [ ]:
df_lep["эксплуатирующая организация"] = df_lep["эксплуатирующая организация"].map(org_mapping)

In [ ]:
df_lep["эксплуатирующая организация"].unique()

In [ ]:
target_lep = 'класс'
target_lep_mapping = {'Кабельные ЛЭП': 0, 
                  'ЛЭП': 1}

df_lep[target_lep] = df_lep[target_lep].map(target_lep_mapping)

In [ ]:
df_lep.to_excel('data/Общий файл (протяженность, напряжение).xlsx', index=False)

In [ ]:
df_lep = pd.read_excel('Общий файл (протяженность, напряжение).xlsx')
df_lep.head()

In [ ]:
df_lep.info()

In [ ]:
X_lep = df_lep.drop(columns=[target_lep])
y_lep = df_lep[target_lep]

# Исследовательский анализ

In [ ]:
num_cols_lep = ["стоимость в ценах 2023 года без НДС, руб", "протяженность, м", "напряжение, кв", "лет в эксплуатации"]

plt.figure(figsize=(5, len(num_cols_lep) * 3))

for i, col in enumerate(num_cols_lep, 1):
    plt.subplot(len(num_cols_lep), 1, i)
    sns.histplot(df_lep[col], kde=True)
    plt.title(f'Распределение {col}')
    plt.xlabel(col)
    plt.ylabel('Количество')

plt.tight_layout()
plt.show()

In [ ]:
num_cols_lep = ["стоимость в ценах 2023 года без НДС, руб", "протяженность, м", "напряжение, кв", "лет в эксплуатации"]

sns.pairplot(df_lep[num_cols_lep], diag_kind="kde")
plt.suptitle("Диаграмма рассеяния", y=1.02)
plt.show()

In [ ]:
num_cols_lep = [
    "стоимость в ценах 2023 года без НДС, руб",
    "протяженность, м",
    "напряжение, кв",
    "лет в эксплуатации"
]

df_corr = df_lep[num_cols_lep].copy()

rename_dict = {
    "стоимость в ценах 2023 года без НДС, руб": "Стоимость",
    "протяженность, м": "Протяженность",
    "напряжение, кв": "Напряжение",
    "лет в эксплуатации": "Возраст"
}

df_corr = df_corr.rename(columns=rename_dict)

corr_spearman = df_corr.corr(method="spearman")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Корреляционная матрица")
plt.show()

In [ ]:
plt.figure(figsize=(8, 12))

plt.subplot(3, 1, 1)
top10_org_lep = df_lep['эксплуатирующая организация'].value_counts().nlargest(5).index
sns.countplot(x=df_lep['эксплуатирующая организация'], order=top10_org_lep)
plt.title("Распределение эксплуатирующих организаций (топ 5)")
plt.ylabel('Количество')

plt.subplot(3, 1, 2)
top10_activity = df_lep['вид деятельности'].value_counts().nlargest(5).index
sns.countplot(x=df_lep['вид деятельности'], order=top10_activity)
plt.title("Распределение вида деятельности")
plt.ylabel('Количество')

plt.subplot(3, 1, 3)
top10_subject_lep = df_lep['субъект РФ'].value_counts().nlargest(5).index
sns.countplot(x=df_lep['субъект РФ'], order=top10_subject_lep)
plt.title("Распределение по субъектам (топ 5)")
plt.ylabel('Количество')

plt.tight_layout()
plt.show()


In [ ]:
profile_lep = ProfileReport(df_lep, title="EDA отчёт", explorative=True)
report_lep = ProfileReport(df_lep, title="Первичный исследовательский анализ", explorative=True)
profile_lep.to_file('data/eda_lep_report.html')
report_lep.to_notebook_iframe()

# Обучение

In [ ]:
num_lep_features = X_lep.select_dtypes(include=['int32', 'int64', 'float64']).columns.tolist()
cat_lep_features = X_lep.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_lep_transformer = StandardScaler()
categorical_lep_transformer = OneHotEncoder(handle_unknown='ignore')
preprocessor_lep = ColumnTransformer(
    transformers=[
        ('num', numerical_lep_transformer, num_lep_features),
        ('cat', categorical_lep_transformer, cat_lep_features)
    ])

In [ ]:
X_lep_train, X_lep_test, y_lep_train, y_lep_test = train_test_split(
    X_lep, y_lep, test_size=0.2, random_state=123, stratify=y_lep)

In [ ]:
logreg_lep = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  
    random_state=123
)

pipeline_logreg_lep = ImbPipeline([
    ('preprocessor', preprocessor_lep),
    ('smote', SMOTE(random_state=123)),
    ('classifier', logreg_lep)
])

param_grid_logreg_lep = {
    'classifier__solver': ['liblinear', 'saga'], 
    'classifier__penalty': ['l1', 'l2', 'elasticnet'],  
    'classifier__C': [0.1, 1, 10],  
    'classifier__l1_ratio': [0, 0.5, 0.9],  
    'smote__k_neighbors': [2, 3, 5]  
}

cv_logreg_lep = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_logreg_lep = GridSearchCV(
    pipeline_logreg_lep,
    param_grid_logreg_lep,
    scoring='roc_auc', 
    cv=cv_logreg_lep,
    n_jobs=-1,
    verbose=1
)

grid_logreg_lep.fit(X_lep_train, y_lep_train)

In [ ]:
print("Лучшие параметры:", grid_logreg_lep.best_params_)
print("ROC-AUC (train):", grid_logreg_lep.best_score_)

y_logreg_lep_pred_proba = grid_logreg_lep.predict_proba(X_lep_test)[:, 1]
roc_logreg_lep_test = roc_auc_score(y_lep_test, y_logreg_lep_pred_proba)

print("ROC-AUC (test):", roc_logreg_lep_test)
y_logreg_lep_pred = grid_logreg_lep.predict(X_lep_test)

cm = confusion_matrix(y_lep_test, y_logreg_lep_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_lep_mapping.keys())
disp.plot(cmap='Blues', values_format='d')
plt.title("Матрица ошибок LogisticRegression")
plt.show()

print(classification_report(y_lep_test, y_logreg_lep_pred, labels = grid_logreg_lep.classes_))

In [ ]:
knn_lep = ImbPipeline([
    ('preprocessor', preprocessor_lep),
    ('smote', SMOTE(random_state=123)),
    ('classifier', KNeighborsClassifier())
])

param_grid_knn_lep = {
    'classifier__n_neighbors': [3, 5, 7, 10],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan'],
    'smote__k_neighbors': [2, 3, 5]
}

cv_knn_lep = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_knn_lep = GridSearchCV(
    knn_lep,
    param_grid_knn_lep,
    scoring='roc_auc',
    cv=cv_knn_lep,
    n_jobs=-1,
    verbose=1
)

grid_knn_lep.fit(X_lep_train, y_lep_train)

In [ ]:
print("Лучшие параметры:", grid_knn_lep.best_params_)
print("ROC-AUC (train):", grid_knn_lep.best_score_)

y_knn_lep_pred_proba = grid_knn_lep.predict_proba(X_lep_test)[:, 1]
roc_knn_lep_test = roc_auc_score(y_lep_test, y_knn_lep_pred_proba)

print("ROC-AUC (test):", roc_knn_lep_test)
y_knn_lep_pred = grid_knn_lep.predict(X_lep_test)

cm = confusion_matrix(y_lep_test, y_knn_lep_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_lep_mapping.keys())
disp.plot(cmap='Blues', values_format='d')
plt.title("Матрица ошибок KNN")
plt.show()

print(classification_report(y_lep_test, y_knn_lep_pred, labels = grid_knn_lep.classes_))

In [ ]:
num_features = num_lep_features

cat_encoder = preproc.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(cat_lep_features)

feature_names = np.concatenate([num_features, cat_features])

X_transformed = preproc.transform(X_lep_test)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names)

def knn_predict_proba(X):
    return clf.predict_proba(X)[:, 1]

background = shap.sample(X_transformed, random_state=123)

explainer = shap.KernelExplainer(
    knn_predict_proba,
    background
)

X_explain = X_transformed

shap_vals = explainer.shap_values(
    X_explain,
    nsamples=100
)

In [ ]:
dt_lep = ImbPipeline(steps=[
    ('preprocessor', preprocessor_lep),
    ('smote', SMOTE(random_state=123)),
    ('classifier', DecisionTreeClassifier(random_state=123))
])

param_grid_dt_lep = {
    "classifier__max_depth": [5, 10, 15],
    "classifier__min_samples_split": [5, 10, 15],
    "classifier__min_samples_leaf": [2, 4, 6],
    "classifier__criterion": ["gini", "entropy"],
    "smote__k_neighbors": [2, 3, 5]
}

cv_dt_lep = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_dt_lep = GridSearchCV(
    dt_lep,
    param_grid_dt_lep,
    scoring='roc_auc',
    cv=cv_dt_lep,
    n_jobs=-1,
    verbose=1
)

grid_dt_lep.fit(X_lep_train, y_lep_train)

In [ ]:
print("Лучшие параметры:", grid_dt_lep.best_params_)
print("ROC-AUC (train):", grid_dt_lep.best_score_)

y_dt_lep_pred_proba = grid_dt_lep.predict_proba(X_lep_test)[:, 1]
roc_dt_lep_test = roc_auc_score(y_lep_test, y_dt_lep_pred_proba)

print("ROC-AUC (test):", roc_dt_lep_test)
y_dt_lep_pred = grid_dt_lep.predict(X_lep_test)

cm = confusion_matrix(y_lep_test, y_dt_lep_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_lep_mapping.keys())
disp.plot(cmap='Blues', values_format='d')
plt.title("Матрица ошибок Decision Tree")
plt.show()

print(classification_report(y_lep_test, y_dt_lep_pred, labels = grid_dt_lep.classes_))

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_lep_train.copy()
X_test_scaled = X_lep_test.copy()

X_train_scaled[num_lep_features] = scaler.fit_transform(
    X_train_scaled[num_lep_features]
)

X_test_scaled[num_lep_features] = scaler.transform(
    X_test_scaled[num_lep_features]
)

for c in cat_lep_features:
    X_train_scaled[c] = X_train_scaled[c].astype(str)
    X_test_scaled[c] = X_test_scaled[c].astype(str)
    
cat_idx = [X_train_scaled.columns.get_loc(c) for c in cat_lep_features]

smote_nc = SMOTENC(
    categorical_features=cat_idx,
    random_state=123
)

X_res, y_res = smote_nc.fit_resample(
    X_train_scaled,
    y_lep_train
)

model_cb_lep = CatBoostClassifier(
    random_state=123,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=0
)

param_grid_cb_lep = {
    "depth": [3, 5, 10],
    "learning_rate": [0.03, 0.05],
    "iterations": [100, 200, 300],
    "l2_leaf_reg": [2, 3, 5]
}

cv_cb_lep = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

grid_cb_lep = GridSearchCV(
    estimator=model_cb_lep,
    param_grid=param_grid_cb_lep,
    scoring="roc_auc",
    cv=cv_cb_lep,
    n_jobs=-1,
    verbose=1
)

grid_cb_lep.fit(
    X_res,
    y_res,
    cat_features=cat_idx
)

In [ ]:
print("Лучшие параметры:", grid_cb_lep.best_params_)
print("ROC-AUC (train):", grid_cb_lep.best_score_)

y_cb_lep_pred_proba = grid_cb_lep.predict_proba(X_test_scaled)[:, 1]
roc_cb_lep_test = roc_auc_score(y_lep_test, y_cb_lep_pred_proba)
print("ROC-AUC (test):", roc_cb_lep_test)

y_cb_lep_pred = grid_cb_lep.predict(X_test_scaled)

cm = confusion_matrix(y_lep_test, y_cb_lep_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Матрица ошибок — CatBoost")
plt.show()

print(classification_report(y_lep_test, y_cb_lep_pred))

In [ ]:
models_lep_probs = {
    'KNN': y_knn_lep_pred_proba,
    'Logreg': y_logreg_lep_pred_proba,
    'Decision Tree': y_dt_lep_pred_proba,
    'CatBoost': y_cb_lep_pred_proba
}

fig = go.Figure()

for model_name, y_proba in models_lep_probs.items():
    fpr, tpr, _ = roc_curve(y_lep_test, y_proba)
    auc = roc_auc_score(y_lep_test, y_proba)
    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode='lines',
        name=f'{model_name} (AUC = {auc:.3f})',
        line=dict(width=3)
    ))
    
fig.add_trace(go.Scatter(
    x=[0,1],
    y=[0,1],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Random'
))

fig.update_layout(
    title='Сравнение ROC-кривых для всех моделей',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_white',
    height=750,
    width=750,
    font=dict(size=14),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [ ]:
results = []

def evaluate_model(
    name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    cat_features=None
):
    start = perf_counter()

    if cat_features is not None:
        model.fit(X_train, y_train, cat_features=cat_features)
    else:
        model.fit(X_train, y_train)

    train_time = perf_counter() - start

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
    else:
        y_proba = None

    auc = None
    if y_proba is not None:
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])

        elif y_proba.ndim == 2 and y_proba.shape[1] > 2:
            auc = roc_auc_score(
                y_test,
                y_proba,
                multi_class="ovr",
                average="weighted"
            )
        else:
            auc = roc_auc_score(y_test, y_proba)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "model": name,
        "train_time_sec": round(train_time, 3),
        "AUC": auc,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

In [ ]:
evaluate_model("Logistic Regression", grid_logreg_lep, X_lep_train, y_lep_train, X_lep_test, y_lep_test)
evaluate_model("KNN", grid_knn_lep, X_lep_train, y_lep_train, X_lep_test, y_lep_test)
evaluate_model("Decision Tree", grid_dt_lep, X_lep_train, y_lep_train, X_lep_test, y_lep_test)
evaluate_model("CatBoost", grid_cb_lep, X_res, y_res, X_test_scaled, y_lep_test, cat_features=cat_idx)

In [ ]:
df_metrics = pd.DataFrame(results)
df_metrics

In [ ]:
feature_names = (
    grid_logreg_lep.best_estimator_["preprocessor"]
    .get_feature_names_out()
)

coefs_logreg = np.abs(grid_logreg_lep.best_estimator_["classifier"].coef_).mean(axis=0)

fi_logreg = pd.DataFrame({
    "feature": feature_names,
    "importance": coefs_logreg
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(fi_logreg["feature"].head(10), fi_logreg["importance"].head(10))
plt.title("Важность признаков - Logistic Regression")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
best_knn = grid_knn_lep.best_estimator_
clf = best_knn.named_steps["classifier"]
preproc = best_knn.named_steps['preprocessor']

preproc.fit(X_lep_train)

X_test_transformed = preproc.transform(X_lep_test)

scoring_metric = "roc_auc" if len(np.unique(y_lep_test)) == 2 else "roc_auc_ovr"

X_test_dense = X_test_transformed.toarray()

r = permutation_importance(
    estimator=clf,  
    X=X_test_dense,
    y=y_lep_test,
    n_repeats=10,
    random_state=123,
    scoring=scoring_metric
)

feature_names = preproc.get_feature_names_out()

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": r.importances_mean
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 8))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Пермутационная важность признаков — KNN")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_vals,
    X_transformed,
    plot_type="bar",
    show=True
)

In [ ]:
shap.summary_plot(
    shap_vals,
    X_transformed
)

In [ ]:
best_tree = grid_dt_lep.best_estimator_
clf = best_tree.named_steps["classifier"]
preproc = best_tree.named_steps['preprocessor']

feature_names = preproc.get_feature_names_out()
importances = clf.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Важность признаков — Decision Tree")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
clf = grid_cb_lep.best_estimator_
importances = clf.get_feature_importance()
feature_names = X_train_scaled.columns
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(fi["feature"], fi["importance"])
plt.title("Важность признаков — CatBoost")
plt.gca().invert_yaxis()
plt.show()